# QVM Quant Stock Screener — Google Colab

Every evening, billion-dollar quant funds run momentum, quality, and profitability screens across the market. This notebook runs the same stack on a 500-stock universe and produces:

1. **Market regime score (0–100)** — trend, volatility, credit, breadth, sentiment, safe-haven
2. **Top 30 ranked stocks** — 7 academic factors + Piotroski F-Score gate
3. **AI-generated bear case** — DeepSeek reads transcripts/news/analyst targets
4. **Regime-aware allocation** — fractional-share portfolio sized to current regime

### Setup
Add your FMP and DeepSeek keys to Colab Secrets (key icon in the left sidebar):
- `FMP_API_KEY` (optional — yfinance fallback if missing)
- `DEEPSEEK_API_KEY` (optional — template bear case if missing)

Both keys are optional. The pipeline runs end-to-end without them.

In [ ]:
# Clone the repo and install requirements.
!git clone https://github.com/alexwagman3/public-qvm-quant-stock-trading.git /content/qvm 2>/dev/null || true
%cd /content/qvm
!pip install -q -r requirements.txt

In [ ]:
# Load API keys from Colab Secrets (skip cleanly if not set).
import os
try:
    from google.colab import userdata
    for key in ('FMP_API_KEY', 'DEEPSEEK_API_KEY'):
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    pass
print('FMP_API_KEY set:', bool(os.environ.get('FMP_API_KEY')))
print('DEEPSEEK_API_KEY set:', bool(os.environ.get('DEEPSEEK_API_KEY')))

In [ ]:
# Run the full pipeline. Takes a few minutes.
!python main.py

In [ ]:
# Display the HTML report inline.
from IPython.display import HTML
with open('output/report-fragment.html') as f:
    HTML(f.read())

In [ ]:
# Inspect the allocation JSON.
import json
with open('output/allocation.json') as f:
    alloc = json.load(f)
print(f"Regime: {alloc['regime']} (score {alloc.get('regime_score')})")
print(f"Capital: ${alloc['capital']:,.0f}  |  Equity: {alloc['equity_pct']*100:.0f}%  |  Positions: {alloc['n_positions']}")
for p in alloc['positions']:
    print(f"  {p['ticker']:>6}  ${p['latest_price']:>8.2f}  {p['shares']:>9.3f} shares  ${p['position_value']:>10,.2f}  {p['weight_pct']:>5.1f}%")